<a href="https://colab.research.google.com/github/YUFEIFUT/Demos/blob/main/colab%E4%BD%9C%E4%B8%BAtf_serving%E7%9A%84%E6%B5%8B%E8%AF%95%E6%9C%8D%E5%8A%A1%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 安装 TensorFlow Serving
此单元格将添加 TensorFlow Serving 的软件包源，并安装模型服务器。

In [1]:
# 添加 TensorFlow Serving 的 APT 软件源
!echo "deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal" | tee /etc/apt/sources.list.d/tensorflow-serving.list && \
curl https://storage.googleapis.com/tensorflow-serving-apt/tensorflow-serving.release.pub.gpg | apt-key add -

# 更新软件列表并安装 tensorflow-model-server
!apt update
!apt-get install tensorflow-model-server

deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100  2943  100  2943    0     0   6508      0 --:--:-- --:--:-- --:--:--  6496
OK
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://storage.googleapis.com/tensorflow-serving-apt stable InRelease [3,026 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://cli.github.com/packages stable/main am

安装完成后，您可以通过运行以下命令来确认安装的版本。

In [2]:
# 检查 TensorFlow Model Server 的版本
!tensorflow_model_server --version

TensorFlow ModelServer: 2.20.0+dev.sha.bc7e9d2
TensorFlow Library: 2.20.0-dev0+selfbuilt


### 下载测试模型数据
我们将克隆 TensorFlow Serving 的仓库以获取官方提供的测试模型（如 `half_plus_two`）。

In [3]:
# 克隆仓库（如果目录已存在则跳过）
import os
if not os.path.exists('serving'):
    !git clone https://github.com/tensorflow/serving

# 设置测试数据路径
import os
testdata_path = os.path.join(os.getcwd(), "serving/tensorflow_serving/servables/tensorflow/testdata")
model_path = os.path.join(testdata_path, "saved_model_half_plus_two_cpu")

print(f"模型路径: {model_path}")

Cloning into 'serving'...
remote: Enumerating objects: 40235, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 40235 (delta 158), reused 86 (delta 78), pack-reused 39989 (from 3)
Receiving objects: 100% (40235/40235), 19.81 MiB | 18.38 MiB/s, done.
Resolving deltas: 100% (32773/32773), done.
模型路径: /content/serving/tensorflow_serving/servables/tensorflow/testdata/saved_model_half_plus_two_cpu


### 启动 TensorFlow Serving 服务
由于 Colab 不方便运行 Docker 容器，我们直接使用已安装的 `tensorflow_model_server` 在后台启动服务。

In [4]:
# 使用 nohup 在后台启动模型服务器，监听 8501 端口
os.environ['MODEL_PATH'] = model_path

!nohup tensorflow_model_server \
    --rest_api_port=8501 \
    --model_name=half_plus_two \
    --model_base_path=$MODEL_PATH > server.log 2>&1 &

print("TensorFlow Serving 正在后台启动... 日志将写入 server.log")

TensorFlow Serving 正在后台启动... 日志将写入 server.log


### 验证服务是否成功启动
等待几秒钟后，我们可以检查日志或尝试发送一个预测请求。

In [5]:
# 检查日志最后几行，看是否加载成功
!tail server.log

# 发送一个测试请求
!curl -d '{"instances": [1.0, 2.0, 5.0]}' -X POST http://localhost:8501/v1/models/half_plus_two:predict

2026-06-18 05:18:13.459626: I external/org_tensorflow/tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 35174 microseconds.
2026-06-18 05:18:13.459786: I tensorflow_serving/servables/tensorflow/saved_model_warmup_util.cc:85] No warmup data file found at /content/serving/tensorflow_serving/servables/tensorflow/testdata/saved_model_half_plus_two_cpu/00000123/assets.extra/tf_serving_warmup_requests
2026-06-18 05:18:13.606522: I tensorflow_serving/core/loader_harness.cc:104] Successfully loaded servable version {name: half_plus_two version: 123}
I0000 00:00:1781759893.607225    2820 server_core.cc:537] Finished adding/updating models
2026-06-18 05:18:13.608267: I tensorflow_serving/model_servers/server.cc:124] Using InsecureServerCredentials
2026-06-18 05:18:13.608447: I tensorflow_serving/model_servers/server.cc:395] Profiler service is enabled
2026-06-18 05:18:13.609198: I tensorflow_serving/model_servers/server.cc:430] Running gRPC Mo

### 使用 ngrok 暴露公网访问接口
我们将安装 `pyngrok` 并通过它将本地的 8501 端口映射到公网。

In [6]:
# 安装 pyngrok
!pip install pyngrok

请在下方填入您的 ngrok Authtoken（可以从 ngrok 官网控制台获取）。

In [12]:
from pyngrok import ngrok
from google.colab import userdata
import os

try:
    # 从 Colab Secrets 中安全读取 Token
    # 请确保您已在左侧“钥匙”面板添加了名为 NGROK_TOKEN 的变量
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')

    # 尝试关闭现有的 ngrok 进程以防冲突
    ngrok.kill()
    # 设置 Token
    ngrok.set_auth_token(NGROK_TOKEN)

    # 创建 HTTP 隧道指向 TensorFlow Serving 的 8501 端口
    tunnel = ngrok.connect(8501, "http")
    public_url = tunnel.public_url
    print(f"TensorFlow Serving 公网访问地址: {public_url}")
except userdata.SecretNotFoundError:
    print("❌ 错误：未在 Colab Secrets 中找到 NGROK_TOKEN。")
    print("请点击左侧钥匙图标，添加 Name 为 NGROK_TOKEN 的密钥并开启访问权限。")
except Exception as e:
    print(f"发生错误: {e}")

TensorFlow Serving 公网访问地址: https://footbath-naming-impaired.ngrok-free.dev


### 外部访问示例
现在您可以复制上面的 `public_url` 地址，在您的本地终端或其他代码中使用它。示例：

In [11]:
# 确保 public_url 已成功定义后再进行打印
if 'public_url' in locals():
    print(f"您可以尝试在您自己电脑的终端运行以下命令来测试远程访问:")
    print(f"curl -d '{{\"instances\": [1.0, 2.0, 5.0]}}' -X POST {public_url}/v1/models/half_plus_two:predict")
else:
    print("❌ 公网隧道尚未成功建立。请先在上面的单元格中填入正确的 Token 并运行成功。")

您可以尝试在您自己电脑的终端运行以下命令来测试远程访问:
curl -d '{"instances": [1.0, 2.0, 5.0]}' -X POST https://footbath-naming-impaired.ngrok-free.dev/v1/models/half_plus_two:predict


windows上调用

curl -X POST https://footbath-naming-impaired.ngrok-free.dev/v1/models/half_plus_two:predict -H "Content-Type: application/json" -d "{\"instances\": [1.0, 2.0, 5.0]}"